# Baseline Modeling

This notebook is primarily about train-plus-validation experimentation, but it now also includes the already-frozen final hold-out comparison section for the logistic, Random Forest, and XGBoost model families. The early sections remain validation-only; the later final comparison section should be treated as benchmark reporting, not tuning.


In [4]:
import pandas as pd

from src.train_baseline_models import run_validation_workflow

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

results = run_validation_workflow()
results["summary_text_v2"]


ModuleNotFoundError: No module named 'src'

## Hold-Out Policy

All feature selection, ablation work, and model tuning below were done on the train and validation splits. The hold-out test period was only used later for the frozen final model-family evaluation and should not be reused for tuning.

Methodology note: after the 2026 audit pass, every modeling result in this notebook is understood as using the **post-2009 modeling universe only**. The full historical dataset remains in the repo for EDA, but the predictive target is treated as reliable only from `2010-01-01` onward because earlier rows show a corner-assignment regime break.



## Why The Old All-Core Model Was Noisy

The earlier all-core model carried several overlapping totals, efficiencies, and index-style features. That can create sign instability and make coefficients hard to trust. The updated workflow now applies train-only feature cleanup before comparing models:
- high-correlation filtering
- near-constant feature filtering
- missingness-based filtering

The goal is not to maximize brute-force feature count, but to keep a leaner and more interpretable feature space that still improves validation signal.


In [ ]:
high_corr = pd.read_csv("outputs/high_correlation_pairs.csv")
low_variance = pd.read_csv("outputs/low_variance_features.csv")

high_corr


In [ ]:
low_variance


## Validation Feature Group Comparison

This is still the quickest way to see whether entire feature families help. Lower log loss is the main priority, with ROC AUC and Brier score as secondary checks.


In [ ]:
feature_group_comparison = pd.read_csv("outputs/validation_feature_group_comparison.csv")
feature_group_comparison


## Regularized Logistic Regression Comparison

The updated workflow compares unregularized, L2, L1, and elastic-net logistic regression on validation only. This helps reduce coefficient instability and test whether a smaller or sparser model is enough to beat the previous benchmark.


In [ ]:
model_type_comparison = pd.read_csv("outputs/validation_model_type_comparison.csv")
model_type_comparison.head(20)


## One-Feature Validation Tests

These one-feature models help identify signals that hold up even without support from the rest of the feature set. This is useful when the multivariate model is noisy.


In [ ]:
single_feature_results = pd.read_csv("outputs/single_feature_validation_results.csv")
single_feature_results.head(20)


## Leave-One-Feature-Out Ablation

Positive `delta_log_loss` means the model got worse when that feature was removed, so the feature was helping.


In [ ]:
lofo = pd.read_csv("outputs/leave_one_feature_out_ablation.csv")
lofo.head(20)


## Leave-One-Group-Out Ablation

This is the group-level version of ablation. It helps answer which feature families still matter after filtering and how much each family contributes to the full validation model.


In [ ]:
logo = pd.read_csv("outputs/leave_one_group_out_ablation.csv")
logo


## Forward Selection Path

Forward selection starts from the experience baseline and greedily adds whichever feature or feature group most improves validation log loss at each step. This keeps the search interpretable instead of brute-forcing a huge search space.


In [ ]:
forward_selection_path = pd.read_csv("outputs/forward_selection_path.csv")
forward_selection_path


## Weight-Class Interactions

The earlier subgroup analysis suggested that reach, control, durability, and some opponent-quality signals may behave differently by division. These interaction tests check whether adding a small set of feature-by-weight-class terms helps validation performance without fully splitting the model into separate divisional models.


In [ ]:
weight_class_interactions = pd.read_csv("outputs/weight_class_interaction_results.csv")
weight_class_interactions


## Calibration

Because this project will eventually care about probabilities, not just winner picks, we also check how the best validation model calibrates its predicted probabilities. The decile table below compares predicted win probability to realized red-win frequency.


In [ ]:
calibration_table = pd.read_csv("outputs/validation_calibration_table.csv")
calibration_table


## Most Promising Signals So Far

Based on validation only:
- age remains one of the strongest individual signals
- experience and record context still matter, but some redundant totals can be dropped
- durability-related features became more useful after filtering and forward selection
- per-fight volume and accuracy terms appear more useful than many redundant raw totals
- weight-class interactions gave only a very small gain in this pass, so they are worth monitoring but not yet a major upgrade
- regularization helped the reduced forward-selected feature set edge past the previous benchmark

The strongest current validation model is now taken from the regularized comparison table rather than from the original all-core unregularized model.


## Final Model Family Comparison

The hold-out test set has now been used for the frozen logistic, Random Forest, and XGBoost models. At this point the test set is no longer untouched, so these results should be treated as the final benchmark comparison rather than a tuning loop.

That final comparison is therefore a comparison of frozen model families on the **2010+ hold-out test era**, not on the full 1994+ fight history.



In [ ]:
final_model_comparison = pd.read_csv('outputs/final_model_test_comparison.csv')
final_model_comparison


Probability prediction takeaway:
- lower `log_loss` and lower `brier_score` are better calibrated / sharper probability outputs
- higher `roc_auc` is better ranking

Classification takeaway:
- higher `accuracy` is better hard 0.5-threshold classification


## Final Logistic Visuals

These plots summarize the frozen logistic benchmark without changing any tuning choices.


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import json

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
figures_dir = repo_root / 'outputs' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

logistic_coef = pd.read_csv(repo_root / 'outputs' / 'final_logistic_coefficients.csv')
logistic_calibration = pd.read_csv(repo_root / 'outputs' / 'final_logistic_calibration_table.csv')


### Final Logistic Coefficients

This bar chart makes the final logistic model easier to read than a plain coefficient table. Positive values favor red, negative values favor blue.


In [ ]:
coef_plot = logistic_coef.sort_values('coefficient')
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['tab:red' if value < 0 else 'tab:blue' for value in coef_plot['coefficient']]
ax.barh(coef_plot['feature'], coef_plot['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Final Logistic Coefficients')
ax.set_xlabel('Coefficient')
fig.tight_layout()
fig.savefig(figures_dir / '03_final_logistic_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()


### Logistic Calibration Plot

This compares average predicted probability to realized win rate by decile on the frozen hold-out test set. The diagonal is perfect calibration.


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
ax.plot(logistic_calibration['mean_predicted_probability'], logistic_calibration['observed_red_win_rate'], marker='o', linewidth=2)
ax.set_title('Logistic Calibration (Test)')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Observed red win rate')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(figures_dir / '03_final_logistic_calibration.png', dpi=150, bbox_inches='tight')
plt.show()


## Walk-Forward Probability Benchmark

The frozen logistic hold-out evaluation above is still the official benchmark section for this model family. Separately, the project now includes a pre-test walk-forward research workflow that compares model probabilities against market no-vig probabilities and calibrated Random Forest variants across annual expanding folds. That workflow does not change the frozen final test results here, but it gives us a more realistic view of probability quality over time.


In [ ]:
walk_forward_comparison = pd.read_csv("outputs/walk_forward_model_comparison.csv")
walk_forward_comparison


The main takeaway from that walk-forward study is that the market remained the strongest standalone probability benchmark on pre-test odds-matched fights, while calibration and market blending improved the tree model relative to its raw probabilities. That is a useful complement to the logistic hold-out story because it separates classification/ranking performance from market-relative probability performance.
